In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Dimensions
N_token = 8        
N_atom = 32        
N_seq = 10         
K_temp = 2         
C = 16             

tau_map = torch.tensor([i // 4 for i in range(N_atom)])

# Operators
def L_bias(in_dim, out_dim): return nn.Linear(in_dim, out_dim, bias=True)
def L_hat(in_dim, out_dim): return nn.Linear(in_dim, out_dim, bias=False)

def SinPE(t, dim):
    half_dim = dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
    emb = t.view(-1, 1) * emb.view(1, -1)
    return torch.cat((emb.sin(), emb.cos()), dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w1 = L_bias(dim, dim)
        self.w2 = L_bias(dim, dim)
        self.w3 = L_bias(dim, dim)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class UniversalAttention(nn.Module):
    def __init__(self, dim, has_edge_bias=True):
        super().__init__()
        self.q_proj = L_hat(dim, dim)
        self.k_proj = L_hat(dim, dim)
        self.v_proj = L_hat(dim, dim)
        self.out_proj = L_bias(dim, dim)
        
        self.has_edge_bias = has_edge_bias
        if has_edge_bias:
            self.b_proj = L_hat(dim, 1)

    def forward(self, q_in, k_in, v_in, qk_einsum, wv_einsum, 
                edge_bias_in=None, bias_unsqueeze_dim=None, mask=None):
        
        q = self.q_proj(q_in)
        k = self.k_proj(k_in)
        v = self.v_proj(v_in)
        
        # 1. QK Dot Product via dynamic shape
        logits = torch.einsum(qk_einsum, q, k) / (q.shape[-1] ** 0.5)
        
        # 2. Add Explicit Edge Bias (if utilized)
        if self.has_edge_bias and edge_bias_in is not None:
            b = self.b_proj(edge_bias_in).squeeze(-1)
            if bias_unsqueeze_dim is not None:
                b = b.unsqueeze(bias_unsqueeze_dim)
            logits = logits + b
            
        # 3. Add Spatial/Alignment Mask
        if mask is not None:
            logits = logits + mask
            
        # 4. Softmax (always on the last dynamically calculated dimension)
        w = F.softmax(logits, dim=-1)
        
        # 5. Value Aggregation
        o = torch.einsum(wv_einsum, w, v)
        return self.out_proj(o)

In [49]:
class Module1_FeatureInit(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.emb_atom = nn.Embedding(20, dim)
        self.emb_pos = L_bias(dim, dim)
        self.broadcast_proj = L_bias(dim, dim)
        
        self.atom_attn = UniversalAttention(dim, has_edge_bias=True)
        
        self.transition = SwiGLU(dim)
        self.pool_proj = L_bias(dim, dim)

    def forward(self, F_A, F_B, tau, s_init):
        # 1.1 & 1.2: Init and Broadcast
        a_i = self.emb_atom(F_A) + self.broadcast_proj(F.layer_norm(s_init[tau], (C,)))
        p_ij = self.emb_pos(F_B)
        
        tau_diff = torch.abs(tau.unsqueeze(1) - tau.unsqueeze(0))
        mask_ij = torch.where(tau_diff <= 1, 0.0, float('-inf'))
        a_tilde = F.layer_norm(a_i, (C,))
        
        # 1.3: Atom Self-Attention (2D Attention logic: q_i * k_j -> ij)
        attn_out = self.atom_attn(
            q_in=a_tilde, k_in=a_tilde, v_in=a_tilde,
            qk_einsum='ic,jc->ij', wv_einsum='ij,jc->ic',
            edge_bias_in=p_ij, mask=mask_ij
        )
        a_prime = a_i + attn_out
        
        # 1.4: Atom Transition
        a_next = a_prime + self.transition(F.layer_norm(a_prime, (C,)))
        
        # 1.5: Atom-to-Token Pooling
        # Note: loop for intuitiveness, use torch.scatter_add for practical
        S_i = torch.zeros(N_token, C)
        for i in range(N_token):
            m = (tau == i).float().unsqueeze(-1)
            S_i[i] = self.pool_proj((F.layer_norm(a_next, (C,)) * m).sum(0) / (m.sum() + 1e-6))
            
        return a_next, S_i, p_ij

In [50]:
class Module2_Template(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.feat_proj = L_bias(dim, dim)
        
        # 2.2 Template Pair Stack (Triangle-like attention within the template)
        self.pair_stack = UniversalAttention(dim, has_edge_bias=True)
        
        # 2.3 Cross Attention: Q from Z_ij, K/V from Templates
        self.cross_attn = UniversalAttention(dim, has_edge_bias=False)

    def forward(self, T_raw, Z_ij, align_mask, seq_mask):
        T_k = self.feat_proj(T_raw)
        
        # 2.2 Template Pair Stack Update
        T_tilde_stack = F.layer_norm(T_k, (C,))
        T_k = T_k + self.pair_stack(
            q_in=T_tilde_stack, k_in=T_tilde_stack, v_in=T_tilde_stack,
            qk_einsum='knmc,knmc->knm', wv_einsum='knm,knmc->knmc',
            edge_bias_in=T_tilde_stack, mask=seq_mask
        )
        
        # 2.3 Cross-Template Aggregation
        Z_tilde = F.layer_norm(Z_ij, (C,))
        T_tilde = F.layer_norm(T_k, (C,))
        
        # Q(n,m,c) * K(k,n,m,c) -> n,m,k
        delta_Z = self.cross_attn(
            q_in=Z_tilde, k_in=T_tilde, v_in=T_tilde,
            qk_einsum='nmc,knmc->nmk', wv_einsum='nmk,knmc->nmc',
            mask=align_mask 
        )
        return Z_ij + delta_Z

In [51]:
class Module3_MSA(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.row_attn = UniversalAttention(dim, has_edge_bias=True)
        self.transition = SwiGLU(dim)
        
        self.left_proj, self.right_proj = L_hat(dim, dim), L_hat(dim, dim)
        self.opm_out = L_bias(dim, dim)

    def forward(self, M_si, Z_ij, msa_mask):
        M_tilde = F.layer_norm(M_si, (C,))
        Z_tilde = F.layer_norm(Z_ij, (C,))
        
        # 3.1: MSA Row Attention (Sequence batch 's', Attending between 'i' and 'j')
        attn_out = self.row_attn(
            q_in=M_tilde, k_in=M_tilde, v_in=M_tilde,
            qk_einsum='sic,sjc->sij', wv_einsum='sij,sjc->sic',
            edge_bias_in=Z_tilde, bias_unsqueeze_dim=0 # Broadcast b_ij across seq dim S
        )
        M_si = M_si + attn_out
        M_si = M_si + self.transition(F.layer_norm(M_si, (C,)))
        
        # 3.2: Outer Product Mean
        M_ln = F.layer_norm(M_si, (C,))
        opm = torch.einsum('sic,sjc->ijc', self.left_proj(M_ln), self.right_proj(M_ln)) 
        
        # Divide strictly by valid mask count to prevent gradient dilution
        Z_ij = Z_ij + self.opm_out(opm / (msa_mask.sum(dim=0).unsqueeze(-1) + 1e-6))
        return M_si, Z_ij

In [52]:
class Module4_PairFormer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 4.1 TriMul params
        self.l_proj, self.r_proj = L_bias(dim, dim), L_bias(dim, dim)
        self.g_mul, self.out_mul = L_bias(dim, dim), L_bias(dim, dim)
        
        # 4.2 TriAtt params
        self.tri_attn = UniversalAttention(dim, has_edge_bias=True)
        self.g_att = L_bias(dim, dim)
        self.transition = SwiGLU(dim)

    def triangle_mult(self, Z):
        Zn = F.layer_norm(Z, (C,))
        L = torch.sigmoid(self.l_proj(Zn)) * self.l_proj(Zn)
        R = torch.sigmoid(self.r_proj(Zn)) * self.r_proj(Zn)
        x_ij = torch.einsum('ikc,jkc->ijc', L, R)
        return Z + torch.sigmoid(self.g_mul(Zn)) * self.out_mul(F.layer_norm(x_ij, (C,)))

    def triangle_att(self, Z):
        Zn = F.layer_norm(Z, (C,))
        
        # ijc * ikc -> ijk, Bias broadcast to i dimension
        attn_out = self.tri_attn(
            q_in=Zn, k_in=Zn, v_in=Zn,
            qk_einsum='ijc,ikc->ijk', wv_einsum='ijk,ikc->ijc',
            edge_bias_in=Zn, bias_unsqueeze_dim=0
        )
        return Z + torch.sigmoid(self.g_att(Zn)) * attn_out

    def forward(self, Z):
        # Outgoing & Starting
        Z = self.triangle_att(self.triangle_mult(Z))
        
        # Incoming & Ending (via Transpose)
        Z = self.triangle_att(self.triangle_mult(Z.transpose(0,1))).transpose(0,1)
        
        return Z + self.transition(F.layer_norm(Z, (C,)))

In [53]:
class Module5_Denoising(nn.Module):
    def __init__(self, dim, num_rbf=16):
        super().__init__()
        self.num_rbf = num_rbf
        self.s_proj = L_bias(dim, dim)
        self.z_proj = L_bias(num_rbf, dim) # Maps RBF bins to channel dim
        
        self.time_mlp = nn.Sequential(L_bias(dim, dim), nn.SiLU(), L_bias(dim, dim))
        self.gamma_proj, self.beta_proj = L_bias(dim, dim), L_bias(dim, dim)
        self.z_time_proj = L_bias(dim, dim)
        
        self.cond_attn = UniversalAttention(dim, has_edge_bias=True)
        
        self.trans_mlp = nn.Sequential(L_bias(dim, dim), nn.SiLU(), L_bias(dim, dim))
        self.hidden_proj = L_bias(dim, dim)
        self.delta_x_proj = L_hat(dim, 3)

    def forward(self, X_t, t, s_i, Z_ij, tau):
        # 5.0 Pre-processing (CoM Alignment)
        X_tc = X_t - X_t.mean(0, keepdim=True)
        
        # Expand RBF to multiple kernels (K bins) to ensure smooth 3D gradients
        d_ij = torch.cdist(X_tc, X_tc).unsqueeze(-1)
        c_k = torch.linspace(0, 20, self.num_rbf).view(1, 1, -1)
        E_RBF = torch.exp(- (d_ij - c_k)**2 / 2.0) 
        
        a_ia = self.s_proj(s_i[tau])
        
        # Expand Token Pair Features to Atom Pair Features via tau map
        Z_tilde = Z_ij[tau][:, tau] + self.z_proj(E_RBF)
        
        # 5.1 AdaLN & Timestep Injection
        c_t = self.time_mlp(SinPE(t, C)) # Shape: [1, C]
        
        # Allow [1, C] to naturally broadcast to [N_atom, C]
        gamma = self.gamma_proj(c_t) 
        beta = self.beta_proj(c_t)   
        a_tilde = F.layer_norm(a_ia, (C,)) * (1 + gamma) + beta 
        
        # Use view to safely broadcast temporal conditioning to [N_atom, N_atom, C]
        Z_tilde = Z_tilde + self.z_time_proj(c_t).view(1, 1, C)
        
        # 5.2 Conditioned Atom Attention
        a_ia = a_ia + self.cond_attn(
            q_in=a_tilde, k_in=a_tilde, v_in=a_tilde,
            qk_einsum='ic,jc->ij', wv_einsum='ij,jc->ic',
            edge_bias_in=Z_tilde
        )
        
        # 5.3 Coordinate Update
        a_ia = a_ia + self.trans_mlp(F.layer_norm(a_ia, (C,)))
        dX = self.delta_x_proj(F.silu(self.hidden_proj(a_ia)))
        
        # Predict denoised coordinates and re-center to prevent drift
        return X_tc + dX - dX.mean(0, keepdim=True)

In [54]:
# Mod 1
a_out, s_out, p_ij_out = Module1_FeatureInit(C)(
    torch.randint(0, 20, (N_atom,)), torch.randn(N_atom, N_atom, C), tau_map, torch.randn(N_token, C)
)
print(f"Module 1 Token shape: {s_out.shape}")

# Mod 2
T_raw = torch.zeros(K_temp, N_token, N_token, C)
align_mask = torch.where(torch.rand(N_token, N_token, K_temp) > 0.5, 0.0, float('-inf'))
seq_mask = torch.zeros(K_temp, N_token, N_token) # Dummy sequence mask for Pair Stack
Z_ij_out = Module2_Template(C)(T_raw, torch.randn(N_token, N_token, C), align_mask, seq_mask)
print(f"Module 2 Z_ij shape: {Z_ij_out.shape}")

# Mod 3
M_si_out, Z_ij_out = Module3_MSA(C)(torch.randn(N_seq, N_token, C), Z_ij_out, torch.ones(N_seq, N_token))
print(f"Module 3 Z_ij shape: {Z_ij_out.shape}")

# Mod 4
Z_ij_final = Module4_PairFormer(C)(Z_ij_out)
print(f"Module 4 Z_ij shape: {Z_ij_final.shape}")

# Mod 5
X_t = torch.randn(N_atom, 3) * 5.0
t = torch.tensor([0.5])
X_0_hat = Module5_Denoising(C)(X_t, t, s_out, Z_ij_final, tau_map)
print(f"Module 5 Final Coords shape: {X_0_hat.shape}")

Module 1 Token shape: torch.Size([8, 16])
Module 2 Z_ij shape: torch.Size([8, 8, 16])
Module 3 Z_ij shape: torch.Size([8, 8, 16])
Module 4 Z_ij shape: torch.Size([8, 8, 16])
Module 5 Final Coords shape: torch.Size([32, 3])
